In [28]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
import seaborn as sns

Считаем датасет

In [29]:
customers = pd.read_csv('hseml-group-project-erste-ritter-team/data/raw/olist_customers_dataset.csv')
geolocation = pd.read_csv('hseml-group-project-erste-ritter-team/data/raw/olist_geolocation_dataset.csv')
order_items = pd.read_csv('hseml-group-project-erste-ritter-team/data/raw/olist_order_items_dataset.csv')
order_payments = pd.read_csv('hseml-group-project-erste-ritter-team/data/raw/olist_order_payments_dataset.csv')
orders = pd.read_csv('hseml-group-project-erste-ritter-team/data/raw/olist_orders_dataset.csv')
products = pd.read_csv('hseml-group-project-erste-ritter-team/data/raw/olist_products_dataset.csv')
sellers = pd.read_csv('hseml-group-project-erste-ritter-team/data/raw/olist_sellers_dataset.csv')
category_name_t = pd.read_csv('hseml-group-project-erste-ritter-team/data/raw/product_category_name_translation.csv')
review = pd.read_csv('hseml-group-project-erste-ritter-team/data/raw/olist_order_reviews_dataset.csv')

Смотрим размеры таблиц

In [30]:
print('Размеры таблиц:')
print('Клиенты:',customers.shape)
print('Геолокация:',geolocation.shape)
print('Позиции заказов:',order_items.shape)
print('Платежи:',order_payments.shape)
print('Заказы:',orders.shape)
print('Товары:',products.shape)
print('Продавцы:',sellers.shape)
print('Перевод категорий:',category_name_t.shape)
print('Отзывы:',review.shape)

Размеры таблиц:
Клиенты: (99441, 5)
Геолокация: (1000163, 5)
Позиции заказов: (112650, 7)
Платежи: (103886, 5)
Заказы: (99441, 8)
Товары: (32951, 9)
Продавцы: (3095, 4)
Перевод категорий: (71, 2)
Отзывы: (99224, 7)


Смотрим на содержание таблиц

In [31]:
print('Колонки таблиц:')
print('Клиенты:',customers.columns)
print('Геолокация:',geolocation.columns)
print('Позиции заказов:',order_items.columns)
print('Платежи:',order_payments.columns)
print('Заказы:',orders.columns)
print('Товары:',products.columns)
print('Продавцы:',sellers.columns)
print('Перевод категорий:',category_name_t.columns)
print('Отзывы:',review.columns)

Колонки таблиц:
Клиенты: Index(['customer_id', 'customer_unique_id', 'customer_zip_code_prefix',
       'customer_city', 'customer_state'],
      dtype='str')
Геолокация: Index(['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng',
       'geolocation_city', 'geolocation_state'],
      dtype='str')
Позиции заказов: Index(['order_id', 'order_item_id', 'product_id', 'seller_id',
       'shipping_limit_date', 'price', 'freight_value'],
      dtype='str')
Платежи: Index(['order_id', 'payment_sequential', 'payment_type',
       'payment_installments', 'payment_value'],
      dtype='str')
Заказы: Index(['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp',
       'order_approved_at', 'order_delivered_carrier_date',
       'order_delivered_customer_date', 'order_estimated_delivery_date'],
      dtype='str')
Товары: Index(['product_id', 'product_category_name', 'product_name_lenght',
       'product_description_lenght', 'product_photos_qty', 'product_weight_g',


Посмотрим количество пропусков в таблицах

In [32]:
print('Количество null:')
print('customers:',customers.isnull().sum().sum())
print('geolocation:',geolocation.isnull().sum().sum())
print('order_items:',order_items.isnull().sum().sum())
print('order_payments:',order_payments.isnull().sum().sum())
print('orders:',orders.isnull().sum().sum())
print('products:',products.isnull().sum().sum())
print('sellers:',sellers.isnull().sum().sum())
print('category_name_t:',category_name_t.isnull().sum().sum())
print('review:',review.isnull().sum().sum())

Количество null:
customers: 0
geolocation: 0
order_items: 0
order_payments: 0
orders: 4908
products: 2448
sellers: 0
category_name_t: 0
review: 145903


Видим наличие пропусков в таблицах товаров, заказов и отзывов. И если наличие отзывов объясняется тем, что не каждый пользователь оставляет отзыв на купленный товар. Проверим таблицы orders и products.

In [33]:
print('orders:',orders.isnull().sum())
print('products:',products.isnull().sum())

orders: order_id                            0
customer_id                         0
order_status                        0
order_purchase_timestamp            0
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
order_estimated_delivery_date       0
dtype: int64
products: product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64


| Поле                         | Пропуски |
|------------------------------|----------|
| order_approved_at        | 160      |
| order_delivered_carrier_date          | 1783     |
| order_delivered_customer_date   | 2965     |

Столбец order_approved_at может быть null в случае отмены заказа (order_status = canceled)

Столбец order_delivered_carrier_date может быть null в случае если заказ еще не передан в доставку 

Столбец order_delivered_customer_date может быть null в случае если заказ еще не доехал до клиента 

Вместо заполнения или удаления пропусков создадим новые признаки

In [34]:
#Признак: заказ был доставлен?
orders['is_delivered'] = orders['order_delivered_customer_date'].notna().astype(int)

#Признак: заказ был одобрен?
orders['is_approved'] = orders['order_approved_at'].notna().astype(int)

#Признак: время доставки в днях (только для доставленных)
orders['order_purchase_timestamp'] = pd.to_datetime(orders['order_purchase_timestamp'])
orders['order_delivered_customer_date'] = pd.to_datetime(orders['order_delivered_customer_date'])

orders['delivery_days'] = (
    orders['order_delivered_customer_date'] - orders['order_purchase_timestamp']
).dt.days

#Для недоставленных заказов будет -1
orders['delivery_days'] = orders['delivery_days'].fillna(-1)

#Признак: задержка доставки (факт > плана)
orders['is_delayed'] = (
    (orders['order_delivered_customer_date'] > orders['order_estimated_delivery_date']) & 
    (orders['is_delivered'] == 1)
).astype(int)

| Поле                         | Пропуски |
|------------------------------|----------|
| product_category_name        | 610      |
| product_name_lenght          | 610      |
| product_description_lenght   | 610      |
| product_photos_qty           | 610      |
| product_weight_g             | 2        |
| product_length_cm            | 2        |
| product_height_cm            | 2        |
| product_width_cm             | 2        |

In [35]:
#Пропуски по 610 заполним нулями
products['product_category_name'] = products['product_category_name'].fillna('unknown')
products['product_name_lenght'] = products['product_name_lenght'].fillna(0)
products['product_description_lenght'] = products['product_description_lenght'].fillna(0)
products['product_photos_qty'] = products['product_photos_qty'].fillna(0)

#Заполним медианой пустые колонки размеров
for col in ['product_weight_g', 'product_length_cm', 'product_height_cm', 'product_width_cm']:
    median_val = products[col].median()
    products[col] = products[col].fillna(median_val)

Проверим снова пропуски

In [36]:
print('Количество null:')
print('customers:',customers.isnull().sum().sum())
print('geolocation:',geolocation.isnull().sum().sum())
print('order_items:',order_items.isnull().sum().sum())
print('order_payments:',order_payments.isnull().sum().sum())
print('orders:',orders.isnull().sum().sum())
print('products:',products.isnull().sum().sum())
print('sellers:',sellers.isnull().sum().sum())
print('category_name_t:',category_name_t.isnull().sum().sum())
print('review:',review.isnull().sum().sum())

Количество null:
customers: 0
geolocation: 0
order_items: 0
order_payments: 0
orders: 4908
products: 0
sellers: 0
category_name_t: 0
review: 145903


Видим, что пропуски остались только в отзывах

In [37]:
cust_orders = customers.merge(orders, on = 'customer_id', how = 'left')
cust_geo = customers.merge(geolocation, left_on = 'customer_zip_code_prefix', right_on = 'geolocation_zip_code_prefix', how = 'left')

Построим график распределения клиентов по штатам

In [38]:
state_counts = customers['customer_state'].value_counts()

plt.figure(figsize=(10, 6))
sns.barplot(x=state_counts.index, y=state_counts.values)
plt.title('Распределение клиентов по штатам')
plt.xlabel('Штат')
plt.ylabel('Количество клиентов')
plt.xticks(rotation=45)
plt.show()
plt.savefig('hseml-group-project-erste-ritter-team/data/eda_plots/barplot-stats.png')

Построим график топ-10 городов по количеству клиентов

In [39]:
top_cities = customers['customer_city'].value_counts().head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_cities.values, y=top_cities.index, palette='magma')
plt.title('Топ-10 городов по количеству клиентов')
plt.xlabel('Количество клиентов')
plt.ylabel('Город')
plt.show()
plt.savefig('hseml-group-project-erste-ritter-team/data/eda_plots/top10-cities.png')

C:\Users\Vladislav\AppData\Local\Temp\ipykernel_6484\1927317716.py:4: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(x=top_cities.values, y=top_cities.index, palette='magma')


Построим корреляционную матрицу платежей

In [40]:
orders_payments = orders.merge(order_payments, on='order_id')
numeric_data = orders_payments.select_dtypes(include=['float64', 'int64'])
corr_matrix = numeric_data.corr()

plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, linewidths=.5, fmt='.2f')
plt.title('Корреляционная матрица')
plt.show()
plt.savefig('hseml-group-project-erste-ritter-team/data/eda_plots/corr_matrix.png')